# Starting with rag
## Requirements :
    -langchain-community
    -PyPDF
    -PymuPDF


### loading documents

In [ ]:
# Text Loader
from langchain_community.document_loaders import TextLoader

loader = TextLoader("../doc_files/notes.txt")
content = loader.load()
print(content)


In [ ]:
# Directory Loader
from langchain_community.document_loaders import DirectoryLoader
from langchain_community.document_loaders import JSONLoader

dirLoader = DirectoryLoader(
    "../doc_files/",
    glob="**/*.json",
    loader_cls=JSONLoader,
    loader_kwargs={"jq_schema": ".", "text_content": False},
    show_progress=False
)
content = dirLoader.load()
content

In [ ]:
# Loading Pdf File
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_community.document_loaders import DirectoryLoader

dir_loader = DirectoryLoader(
    "../doc_files/",
    glob="**/*.pdf",
    loader_cls=PyMuPDFLoader,
    loader_kwargs={"extract_images":True, "extract_tables": "markdown"},
    use_multithreading=True
)
contents = dir_loader.load()
contents

In [ ]:
# Create dummy files and write the some content 

import os

storage = {
    "notes.txt": "This is a simple text file content.",
    "config.json": '{"setting": "enabled", "version": 1.0}',
    "script.py": "print('Hello from the script!')",
}

for file, content in storage.items():
    with open(f"../doc_files/{file}", "w", encoding="utf-8") as f:
        f.write(content)


print("content Written Successfully...")

# RAG Pipeline (From Indexing to Vector Db pipleine)
### Requirements: 
    -langchain-community(PyPDFLoader and PyMuPDF)
    -langchain.textsplitter (RecurisveCharacterTextSplitter)
    pathlib

In [ ]:
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path


In [ ]:
# create a function that loads all the pdf file in the dir and returns the whole documents by adding corresponding metadata fields like file_name and filetype
def getPdfDocs(pdfDir):
    allDocs = []
    pobj = Path(pdfDir)
    if not pobj.is_dir():
        print(f"Dir Not Found")
        return None
    pdf_files = pobj.rglob("**/*.pdf")
    # procces the pdf files 
    for pdf_file in pdf_files:
        print(f"Processing :{pdf_file.stem}")
        loader = PyPDFLoader(str(pdf_file))
        docs = loader.load()
        print(f"Loaded {len(docs)} pages")
        for doc in docs:
            doc.metadata["file_name"] = pdf_file.stem
            doc.metadata["file_type"] = pdf_file.suffix
        allDocs.extend(docs)
    return allDocs
all_docs = getPdfDocs("../doc_files/")

In [6]:
# a text splitter function
def split_doc(docs, chunkSize=1000, chunk_overlap=200):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size = chunkSize,
        chunk_overlap = chunk_overlap,
        separators=["\n\n", "\n", " ", ""],
        length_function =len
    )
    splitted_doc = text_splitter.split_documents(docs)
    print(f"splitted {len(docs)} Docs into {len(splitted_doc)}")
    print(f"Content: {splitted_doc[0].page_content[:200]}")
    return splitted_doc
split_doc(all_docs)

splitted 3 Docs into 15
Content: SOWMYA G 
Software Developer | Python & Django Developer | Backend Development 
+91-8147588578 | g34402284@gmail.com | github.com/SowmyaGopal12 | linkedin//Sowmya G 
SUMMARY 
Computer Science undergra


[Document(metadata={'producer': 'Microsoft® Word 2021', 'creator': 'Microsoft® Word 2021', 'creationdate': '2026-08-03T16:14:20+05:30', 'author': 'Un-named', 'moddate': '2026-08-03T16:14:20+05:30', 'source': '..\\doc_files\\SowmyaG_Resume.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'file_name': 'SowmyaG_Resume', 'file_type': '.pdf'}, page_content='SOWMYA G \nSoftware Developer | Python & Django Developer | Backend Development \n+91-8147588578 | g34402284@gmail.com | github.com/SowmyaGopal12 | linkedin//Sowmya G \nSUMMARY \nComputer Science undergraduate skilled in Python, Java, and Django, with hands -on experience building full-stack style applications \nusing object-oriented design and clean UI principles. Experienced building CRUD -driven applications with structured data \nmanagement, exception handling, and input validation. Strong foundation in Object -Oriented Programming, file handling, and version \ncontrol through Git and GitHub. \nTECHNICAL SKILLS \nLanguages: Pyth

# Embedding and vectordb

In [2]:
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
from sklearn.metrics.pairwise import cosine_similarity
import uuid
from typing import List, Dict, Any, Tuple
import numpy as np
import os

In [13]:
class EmbeddingManager:
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        self.model = None
        self.model_name = model_name
        self._load_model()
    
    def _load_model(self):
        try:
            self.model = SentenceTransformer(self.model_name)
            print(f"Model Dimension : {self.model.get_embedding_dimension()}")
        except Exception as e:
            print(f"Error Loading Model: {self.model_name} : {e}")
            raise ValueError("Model Cannot Be Loaded...")
        
    def generate_embedding(self, texts: List[str]):
        try:
            encodings = self.model.encode(texts, show_progress_bar=True)
            return encodings
        except Exception as e :
            print(e)
            
    def get_embedding_dimension(self):
        if not self.model:
            raise ValueError("Model Doesn't Exist")
        return self.model.get_embedding_dimension()

embedding_manager = EmbeddingManager()
embedding_manager

c:\Users\Vivek\Desktop\agentic_ai\.venv\Lib\site-packages\huggingface_hub\file_download.py:150: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Vivek\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
c:\Users\Vivek\Desktop\agentic_ai\.venv\Lib\site-packages\huggingface_hub\file_download.py

Model Dimension : 384


In [ ]:
# Vector Store
class VectorStore:
    def __init__(self, collection_name: str = "PDF_Collection", persist_dir: str = "./vector_store"):
        self.collection_name = collection_name
        self.persist_dir = persist_dir
        self.collection = None
        self.client = None
        self._load_vectorDB()
    
    # load the colleciton and client
    def _load_vectorDB(self):
        # initialize collection and client
        os.makedirs(self.persist_dir, exist_ok=True)
        try:
            self.client = chromadb.PersistentClient(path=self.persist_dir)
            self.collection = self.client.get_or_create_collection(self.collection_name, metadata={
                "description": "PDF files for RAG"
            })
            print(f"Vector Store initialized successfully {self.collection_name}")
            print(f"Existing Documents in collection {self.collection.count()}")
        except Exception as e:
            print(e)    
    # Add docs to vector store 
    def add_docs(self, docs: List[Any], embeddings: np.ndarray):
        # prepare data 
        doc_ids = []
        doc_contents = []
        doc_embeddings = []
        metadatas = []
        for i, (doc, embedding) in enumerate(zip(docs, embeddings)):
            # unique id for each doc
            uid = f"{uuid.uuid4().hex[:8]}_{i}"
            doc_ids.append(uid)
            
            # prepare page content
            doc_contents.append(doc.page_content)
            
            # embeddings 
            doc_embeddings.append(embedding.tolist())
            
            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata["document_index"] = i
            metadata["content_length"] = len(doc.page_content)
            metadatas.append(metadata)
            
        # add fields to vector Store 
        try:
            self.collection.add(
                ids=doc_ids,
                documents=doc_contents,
                embeddings=doc_embeddings
            )
            print(f"Successfully added {len(docs)} into vector store...")
        except Exception as e :
            print(e)
            
vectorstore = VectorStore()